# Toxic Comment Classification

Multi-label text classification system to identify toxic comments across 6 categories using NLP and machine learning.

## Problem Statement
Identify toxic online comments to help platforms moderate content. The dataset has severe class imbalance (only 9.6% toxic comments overall, with some categories <1%).

## Dataset
- **Source**: Jigsaw Toxic Comment Classification Challenge (Kaggle)
- **Size**: 159,571 comments
- **Labels**: 6 categories (toxic, severe_toxic, obscene, threat, insult, identity_hate)
- **Challenge**: Highly imbalanced multi-label classification

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import multilabel_confusion_matrix, classification_report
from sklearn.metrics import average_precision_score

print("Ready")

Ready


## 1. Data Loading and Exploration

In [40]:
raw_df = pd.read_csv("data/train.csv")
raw_df.shape

(159571, 8)

## 2. Exploratory Data Analysis

### Dataset Overview

In [41]:
raw_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 159571 entries, 0 to 159570
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   id             159571 non-null  str  
 1   comment_text   159571 non-null  str  
 2   toxic          159571 non-null  int64
 3   severe_toxic   159571 non-null  int64
 4   obscene        159571 non-null  int64
 5   threat         159571 non-null  int64
 6   insult         159571 non-null  int64
 7   identity_hate  159571 non-null  int64
dtypes: int64(6), str(2)
memory usage: 9.7 MB


In [42]:
raw_df.describe()

,toxic,severe_toxic,obscene,threat,insult,identity_hate
count,159571.000000,159571.000000,159571.000000,159571.000000,159571.000000,159571.000000
mean,0.095844,0.009996,0.052948,0.002996,0.049364,0.008805
std,0.294379,0.099477,0.223931,0.054650,0.216627,0.093420
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [43]:
raw_df.isna().sum()

id               0
comment_text     0
toxic            0
severe_toxic     0
obscene          0
threat           0
insult           0
identity_hate    0
dtype: int64

In [44]:
df = raw_df.copy()

In [45]:
#df.drop(columns = ['id'], inplace = True)

### Class Distribution Analysis
Understanding imbalance is critical for toxic comment detection.

In [46]:
print(df[['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']].sum())
print(df[['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']].mean() * 100)

toxic            15294
severe_toxic      1595
obscene           8449
threat             478
insult            7877
identity_hate     1405
dtype: int64
toxic            9.584448
severe_toxic     0.999555
obscene          5.294822
threat           0.299553
insult           4.936361
identity_hate    0.880486
dtype: float64


## 3. Data Preparation

### Train-Test Split


In [47]:
target_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
input_col = 'comment_text'

X = df[input_col]
y = df[target_cols]

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

print("Shape of training input columns (X_train):  ",X_train.shape)
print("Shape of test input columns (X_test):       ",X_test.shape)
print("Shape of training target columns (y_train): ",y_train.shape)
print("Shape of test target columns (y_test):      ",y_test.shape)

Shape of training input columns (X_train):   (119678,)
Shape of test input columns (X_test):        (39893,)
Shape of training target columns (y_train):  (119678, 6)
Shape of test target columns (y_test):       (39893, 6)


In [48]:
print(type(y_train))
print(y_train.shape)


<class 'pandas.DataFrame'>
(119678, 6)


## 4. Model Training & Evaluation
### Model 1: Logistic Regression (Baseline)

In [49]:
pipe_lr = Pipeline([('tfidf_lr', TfidfVectorizer(
        ngram_range=(1,2),
        max_features=50000,
        sublinear_tf=True)),
                    ('model_lr', OneVsRestClassifier(LogisticRegression(solver='liblinear',
                                                                         class_weight='balanced',
                                                                           max_iter=1000,
                                                                             random_state=42)))])

pipe_lr.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('tfidf_lr', ...), ('model_lr', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


In [50]:
y_test_pred_lr = pipe_lr.predict(X_test)
y_test_pred_lr

array([[1, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       ...,
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0]], shape=(39893, 6))

In [52]:
multilabel_confusion_matrix(y_test, y_test_pred_lr)

array([[[34326,  1752],
        [  547,  3268]],

       [[38711,   776],
        [   75,   331]],

       [[36945,   805],
        [  266,  1877]],

       [[39570,   218],
        [   28,    77]],

       [[36604,  1278],
        [  267,  1744]],

       [[38782,   754],
        [   99,   258]]])

In [53]:
print(classification_report(y_test, y_test_pred_lr))

              precision    recall  f1-score   support

           0       0.65      0.86      0.74      3815
           1       0.30      0.82      0.44       406
           2       0.70      0.88      0.78      2143
           3       0.26      0.73      0.39       105
           4       0.58      0.87      0.69      2011
           5       0.25      0.72      0.38       357

   micro avg       0.58      0.85      0.69      8837
   macro avg       0.46      0.81      0.57      8837
weighted avg       0.61      0.85      0.71      8837
 samples avg       0.06      0.08      0.07      8837



In [ ]:
y_test_proba_lr = pipe_lr.predict_proba(X_test)

pr_auc = average_precision_score(y_test, y_test_proba_lr, average='macro')
print("LR Macro PR-AUC: ", pr_auc)

LR Macro PR-AUC:  0.6455228868652892


### Model 2: Naive Bayes

In [55]:
pipe_nb = Pipeline([('tfidf_nb', TfidfVectorizer(ngram_range=(1,2),max_features=50000,sublinear_tf=True)),
                    ('model_nb', OneVsRestClassifier(MultinomialNB(alpha=0.1)))])
pipe_nb.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('tfidf_nb', ...), ('model_nb', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


In [56]:
y_test_pred_nb = pipe_nb.predict(X_test)

In [57]:
multilabel_confusion_matrix(y_test, y_test_pred_nb)

array([[[35543,   535],
        [ 1396,  2419]],

       [[39278,   209],
        [  211,   195]],

       [[37373,   377],
        [  777,  1366]],

       [[39765,    23],
        [   91,    14]],

       [[37398,   484],
        [  815,  1196]],

       [[39385,   151],
        [  291,    66]]])

In [58]:
print(classification_report(y_test, y_test_pred_nb))

              precision    recall  f1-score   support

           0       0.82      0.63      0.71      3815
           1       0.48      0.48      0.48       406
           2       0.78      0.64      0.70      2143
           3       0.38      0.13      0.20       105
           4       0.71      0.59      0.65      2011
           5       0.30      0.18      0.23       357

   micro avg       0.75      0.59      0.66      8837
   macro avg       0.58      0.44      0.50      8837
weighted avg       0.74      0.59      0.66      8837
 samples avg       0.05      0.05      0.05      8837



In [59]:
y_test_proba_nb = pipe_nb.predict_proba(X_test)

pr_auc = average_precision_score(y_test, y_test_proba_nb, average='macro')
print("NB Macro PR-AUC: ", pr_auc)


NB Macro PR-AUC:  0.5216166119294597


### Model 3: XGBoost

In [60]:
pipe_xgb = Pipeline([('tfidf_xgb', TfidfVectorizer(ngram_range=(1,2), max_features=50000, sublinear_tf=True)),
                     ('model_xgb', XGBClassifier(scale_pos_weight = 13, eval_metric='aucpr', random_state = 42))])
pipe_xgb.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('tfidf_xgb', ...), ('model_xgb', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


In [61]:
y_test_pred_xgb = pipe_xgb.predict(X_test)

In [62]:
multilabel_confusion_matrix(y_test, y_test_pred_xgb)

array([[[32683,  3395],
        [  556,  3259]],

       [[39099,   388],
        [  173,   233]],

       [[37119,   631],
        [  325,  1818]],

       [[39734,    54],
        [   65,    40]],

       [[36643,  1239],
        [  391,  1620]],

       [[39321,   215],
        [  194,   163]]])

In [63]:
print(classification_report(y_test, y_test_pred_xgb))

              precision    recall  f1-score   support

           0       0.49      0.85      0.62      3815
           1       0.38      0.57      0.45       406
           2       0.74      0.85      0.79      2143
           3       0.43      0.38      0.40       105
           4       0.57      0.81      0.67      2011
           5       0.43      0.46      0.44       357

   micro avg       0.55      0.81      0.65      8837
   macro avg       0.51      0.65      0.56      8837
weighted avg       0.56      0.81      0.66      8837
 samples avg       0.07      0.08      0.07      8837



In [64]:
y_test_proba_xgb = pipe_xgb.predict_proba(X_test)

pr_auc = average_precision_score(y_test, y_test_proba_xgb, average='macro')
print("XGB Macro PR-AUC: ", pr_auc)


XGB Macro PR-AUC:  0.6016906938668789


### Baseline Comparison: Dummy Classifier

In [69]:
from sklearn.dummy import DummyClassifier

pipe_dummy = Pipeline([
    ('tfidf',      TfidfVectorizer()),
    ('model_dummy', OneVsRestClassifier(DummyClassifier(strategy='most_frequent')))
])

pipe_dummy.fit(X_train, y_train)

y_test_pred_dummy  = pipe_dummy.predict(X_test)
y_test_proba_dummy = pipe_dummy.predict_proba(X_test)

print(classification_report(y_test, y_test_pred_dummy))
print("Dummy Macro PR-AUC: ", average_precision_score(y_test, y_test_proba_dummy, average='macro'))


              precision    recall  f1-score   support

           0       0.00      0.00      0.00      3815
           1       0.00      0.00      0.00       406
           2       0.00      0.00      0.00      2143
           3       0.00      0.00      0.00       105
           4       0.00      0.00      0.00      2011
           5       0.00      0.00      0.00       357

   micro avg       0.00      0.00      0.00      8837
   macro avg       0.00      0.00      0.00      8837
weighted avg       0.00      0.00      0.00      8837
 samples avg       0.00      0.00      0.00      8837

Dummy Macro PR-AUC:  0.036919593245264413


## 5. Results Comparison

| Model | Macro F1 | Micro F1 | Macro PR-AUC | Key Insight |
|-------|----------|----------|--------------|-------------|
| Dummy (Baseline) | 0.00 | 0.00 | 0.037 | Predicts majority class only (fails to detect toxic labels) |
| Naive Bayes | 0.50 | 0.66 | 0.522 | More conservative, lower recall on rare labels |
| XGBoost | 0.56 | 0.65 | 0.602 | Balanced performance across labels |
| **Logistic Regression** | **0.57** | **0.69** | **0.646** | **Best overall – strongest PR-AUC & stable across labels** |